# GroupBy and AggregateFunction

1. Total `amount` per `category`, highest first.





7. Show the single biggest order of each city.
8. Count of orders per `city` and `category` with `crosstab`.

In [31]:

import pandas as pd
import numpy as np
import os

DATA_DIR = "/tmp/pandas_class"                  # a folder on the machine, works in Databricks too
os.makedirs(DATA_DIR, exist_ok=True)

np.random.seed(1)
n = 200
price_list = {"Laptop": 55000, "Mouse": 700, "Chair": 4500, "Desk": 9000, "Notebook": 60}
category_list = {"Laptop": "Electronics", "Mouse": "Electronics", "Chair": "Furniture",
                 "Desk": "Furniture", "Notebook": "Stationery"}

products = np.random.choice(list(price_list), n)

sales = pd.DataFrame({
    "order_id": np.arange(1001, 1001 + n),
    "order_date": np.random.choice(pd.date_range("2026-01-01", "2026-06-30"), n),
    "customer": np.random.choice(["Amit", "Sara", "John", "Priya", "Ravi", "Neha"], n),
    "city": np.random.choice(["Pune", "Mumbai", "Delhi", "Chennai"], n),
    "product": products,
    "category": [category_list[p] for p in products],
    "qty": np.random.randint(1, 6, n),
    "price": [price_list[p] for p in products],
})
sales["order_date"] = sales["order_date"].dt.strftime("%Y-%m-%d")
sales.to_csv(f"{DATA_DIR}/sales.csv", index=False)

print("Created:", f"{DATA_DIR}/sales.csv")
print("Rows:", len(sales), " Columns:", len(sales.columns))

Created: /tmp/pandas_class/sales.csv
Rows: 200  Columns: 8


In [32]:
print(sales)

     order_id  order_date customer     city   product     category  qty  price
0        1001  2026-02-04     Ravi     Pune      Desk    Furniture    3   9000
1        1002  2026-04-11     Sara   Mumbai  Notebook   Stationery    3     60
2        1003  2026-02-02    Priya   Mumbai    Laptop  Electronics    4  55000
3        1004  2026-05-28     Neha    Delhi     Mouse  Electronics    5    700
4        1005  2026-01-25     Sara  Chennai      Desk    Furniture    2   9000
..        ...         ...      ...      ...       ...          ...  ...    ...
195      1196  2026-06-01     Sara     Pune     Chair    Furniture    3   4500
196      1197  2026-05-21     Amit    Delhi      Desk    Furniture    3   9000
197      1198  2026-05-26     Sara   Mumbai      Desk    Furniture    1   9000
198      1199  2026-06-11     John    Delhi  Notebook   Stationery    4     60
199      1200  2026-01-11     John    Delhi      Desk    Furniture    3   9000

[200 rows x 8 columns]


In [33]:
sales["amount"] = sales["qty"] * sales["price"]

In [34]:
#Total `amount` per `category`, highest first.
sales.groupby("category")["amount"].sum().sort_values(ascending=False)
# sales.groupby("category")["amount"].sum().sort_values(ascending = False)

category
Electronics    8004700
Furniture      1444500
Stationery        7140
Name: amount, dtype: int64

In [35]:
# 2. Number of orders and total amount per `customer`. Who bought the most?
sales.groupby("customer").agg(
    NoOfOrders= ("order_id","count"),
    TotalAmt = ("amount","sum")
).sort_values("TotalAmt",ascending=False)

,NoOfOrders,TotalAmt
customer,,
Neha,35,2501320
Priya,39,2030880
John,35,1509880
Sara,24,1416600
Ravi,34,1357500
Amit,33,640160


In [36]:
# 3. Average `qty` per `product`, rounded to 2 decimals.
sales.groupby("product")['qty'].mean().round(2)

product
Chair       2.79
Desk        3.14
Laptop      3.27
Mouse       2.88
Notebook    2.70
Name: qty, dtype: float64

In [37]:
# 4. A report per `city` with: number of orders, total sales, average order value and
#    the number of different products sold. Use named aggregation.
city_report = sales.groupby("city", as_index=False).agg(
    NoOrders = ("order_id","count"),
    total_sales = ("amount","sum"),
    AvgOrder = ("amount","mean"),
    ProductSold = ("product","count")
)
city_report["AvgOrder"] = city_report["AvgOrder"].round(0)
city_report

,city,NoOrders,total_sales,AvgOrder,ProductSold
0,Chennai,44,2484840,56474.0,44
1,Delhi,51,2911960,57097.0,51
2,Mumbai,55,2410760,43832.0,55
3,Pune,50,1648780,32976.0,50


In [39]:
sales["month"] = pd.to_datetime(sales["order_date"]).dt.month_name() 
# sales["order_date"].dt.month_name()

In [41]:
# 5. Total sales per `month` per `city` as a pivot table (months as rows, cities as columns).
pd.pivot_table(sales, index="month",columns="city",values="amount",
               aggfunc="sum", fill_value=0)

city,Chennai,Delhi,Mumbai,Pune
month,,,,
April,627680,473180,347600,307900
February,15360,333500,831260,313800
January,219840,428120,203040,49920
June,485820,281060,502300,446700
March,627600,559060,319480,200120
May,508540,837040,207080,330340


In [42]:
# 6. Add a column `pct_of_total` = each order's share of its category total.
sales["pct_of_total"] = (sales["amount"]/sales.groupby("category")["amount"].transform("sum") *100).round(2)

sales[["category", "amount", "pct_of_total"]].head()

,category,amount,pct_of_total
0,Furniture,27000,1.87
1,Stationery,180,2.52
2,Electronics,220000,2.75
3,Electronics,3500,0.04
4,Furniture,18000,1.25
